# 02 — Model Training and Validation

**QM640 | Next-quarter PAT-growth regression | Development stage**

This notebook consumes the exact-quarter, leakage-controlled dataset produced by `01_Data_Preparation_and_Statistical_Tests.ipynb`. It **does not modify the target** or conduct final test-set evaluation. It tunes and selects among a historical median baseline, Linear Regression, Ridge, Random Forest, and Gradient Boosting using **training and validation quarters only**.

**Inputs:** `statistical_outputs/prepared_company_quarters.csv`, `statistical_outputs/handoff_manifest.json`.

**Outputs:** `model_outputs/` containing validation metrics, tuning configurations, validation predictions, fitted candidate models, a frozen selection manifest, and prepared metadata.

**Research integrity:** The held-out `test` rows are excluded from model fitting, tuning, and selection. Fitted artifacts here are subsequently refitted on train+validation by Notebook 03 using the **frozen** selection; Notebook 03 is responsible for final test evaluation. No claim of predictive effectiveness is made until it runs.

## Setup, provenance and split integrity

In [ ]:
from pathlib import Path
import json, hashlib, warnings, platform
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

SEED = 640
SOURCE = Path("statistical_outputs")
if not (SOURCE / "prepared_company_quarters.csv").exists():
    SOURCE = Path("/content/statistical_outputs")
if not (SOURCE / "prepared_company_quarters.csv").exists():
    raise FileNotFoundError("Run notebook 01, then place its statistical_outputs folder beside this notebook (or in /content).")
OUT = SOURCE.parent / "model_outputs"
OUT.mkdir(parents=True, exist_ok=True)
prepared_file = SOURCE / "prepared_company_quarters.csv"
manifest = json.loads((SOURCE / "handoff_manifest.json").read_text())
actual_hash = hashlib.sha256(prepared_file.read_bytes()).hexdigest()
assert actual_hash == manifest["prepared_csv_sha256"], "Prepared data changed since notebook 01."
df = pd.read_csv(prepared_file, low_memory=False)
required = {"COMPANY_KEY","target_q","target_growth_pct","split","INDEX_INDUSTRY","quarter_of_year"}
assert required.issubset(df.columns), sorted(required - set(df.columns))
df = df[df.split.isin(["train","validation","test"])].copy()
assert df.target_growth_pct.notna().all()
assert set(df.split.unique()) == {"train","validation","test"}
qsets = {s: set(df.loc[df.split.eq(s), "target_q"].astype(str)) for s in ["train","validation","test"]}
assert not (qsets["train"] & qsets["validation"] or qsets["train"] & qsets["test"] or qsets["validation"] & qsets["test"])
assert max(qsets["train"]) < min(qsets["validation"]) < min(qsets["test"]), "Chronological ordering invalid."
train = df[df.split.eq("train")].copy()
valid = df[df.split.eq("validation")].copy()
print("Rows:", {s: len(df[df.split.eq(s)]) for s in qsets})
print("Test data are not used for tuning or selection.")

## Feature sets and leakage-safe preprocessing

In [ ]:
reported = ["PAT","REVENUE","TOTAL_INCOME","OTHER_INCOME","PROFIT_BEFORE_TAX",
            "TAX_EXPENSE","EXPENSES","FINANCE_COSTS","EPS_BASIC","EPS_DILUTED"]
engineered = ["pat_growth_pct","revenue_growth_pct","eps_basic_growth_pct",
              "pat_margin_pct","finance_cost_income_pct","expense_income_pct","negative_base_pat"]
categorical = ["INDEX_INDUSTRY","quarter_of_year"]
assert set(reported + engineered + categorical).issubset(df.columns), "Check feature columns from notebook 01."
feature_sets = {"reported_only": reported + categorical,
                "reported_plus_engineered": reported + engineered + categorical}
for c in reported + engineered:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf,-np.inf],np.nan)
train = df[df.split.eq("train")].copy()
valid = df[df.split.eq("validation")].copy()
for c in categorical:
    train[c] = train[c].fillna("Unknown").astype(str)
    valid[c] = valid[c].fillna("Unknown").astype(str)

def make_pipeline(model, features):
    nums = [c for c in features if c not in categorical]
    cats = [c for c in features if c in categorical]
    numeric = Pipeline([("impute",SimpleImputer(strategy="median", add_indicator=False)),
                        ("scale",StandardScaler())])
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # older scikit-learn
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    transforms = [("num",numeric,nums),("cat",encoder,cats)]
    preprocessor = ColumnTransformer(transforms, remainder="drop")
    return Pipeline([("preprocess",preprocessor),("model",model)])

def metrics(y, yhat):
    return {"MAE":float(mean_absolute_error(y,yhat)),
            "RMSE":float(np.sqrt(mean_squared_error(y,yhat))),
            "R2":float(r2_score(y,yhat))}
print("Reported:",len(feature_sets["reported_only"]),"Enhanced:",len(feature_sets["reported_plus_engineered"]))
print("All transformers are fitted inside training folds, not on validation/test rows.")

## Candidate models and hyperparameter search

For transparency, a compact grid is used rather than an expensive uncontrolled search. A **single chronological validation block** selects the primary model by MAE (with RMSE as a tie-breaker). This short panel cannot support claims of stable forecasting performance. The reported-only and enhanced feature sets are both tuned on development data; RQ4 can later use an equal-algorithm contrast with parameters chosen without test access.

The historical median benchmark uses the median target in training data; it is a constant baseline, **not** a per-company forecast.

In [ ]:
candidates = {
 "Historical median": [{"strategy":"median"}],
 "Linear Regression": [{}],
 "Ridge": list(ParameterGrid({"alpha":[1.0,10.0,100.0]})),
 "Random Forest": list(ParameterGrid({"n_estimators":[150],"max_depth":[5,10],"min_samples_leaf":[5,15]})),
 "Gradient Boosting": list(ParameterGrid({"n_estimators":[100,200],"learning_rate":[0.03,0.08],"max_depth":[2],"min_samples_leaf":[10]})),
}
def estimator(name, params):
    if name == "Historical median": return DummyRegressor(**params)
    if name == "Linear Regression": return LinearRegression()
    if name == "Ridge": return Ridge(**params)
    if name == "Random Forest": return RandomForestRegressor(random_state=SEED,n_jobs=-1,**params)
    if name == "Gradient Boosting": return GradientBoostingRegressor(random_state=SEED,**params)
    raise ValueError(name)

rows, predictions, fitted = [], [], {}
for feature_set, cols in feature_sets.items():
    for name, grid in candidates.items():
        for params in grid:
            pipe = make_pipeline(estimator(name,params),cols)
            pipe.fit(train[cols],train.target_growth_pct)
            pred = pipe.predict(valid[cols])
            score = metrics(valid.target_growth_pct,pred)
            key = f"{feature_set}|{name}|{json.dumps(params,sort_keys=True)}"
            rows.append({"key":key,"feature_set":feature_set,"algorithm":name,
                         "params_json":json.dumps(params,sort_keys=True),**score})
            fitted[key] = pipe
            predictions.append(pd.DataFrame({
                "key":key,"COMPANY_KEY":valid.COMPANY_KEY.to_numpy(),
                "target_q":valid.target_q.to_numpy(),
                "actual":valid.target_growth_pct.to_numpy(),"predicted":pred
            }))
trials = pd.DataFrame(rows).sort_values(["MAE","RMSE","key"]).reset_index(drop=True)
assert trials.MAE.notna().all() and np.isfinite(trials[["MAE","RMSE","R2"]]).all().all()
trials.to_csv(OUT/"validation_trials.csv",index=False)
pd.concat(predictions,ignore_index=True).to_csv(OUT/"validation_predictions.csv",index=False)
display(trials.head(15)[["feature_set","algorithm","params_json","MAE","RMSE","R2"]])

## Freeze the model choices before accessing held-out test data

In [ ]:
# Primary selection covers every candidate model on the same validation observations.
winner = trials.iloc[0]
# For RQ3, retain each algorithm's best validation configuration within the enhanced feature set.
best_by_algorithm = (trials.loc[trials.feature_set.eq("reported_plus_engineered")]
                     .sort_values(["MAE","RMSE","key"]).drop_duplicates("algorithm"))
# RQ4 comparison: use the chosen algorithm with best development-only configuration for EACH feature set.
selected_algorithm = str(winner.algorithm)
rq4_by_feature = (trials.loc[trials.algorithm.eq(selected_algorithm)]
                  .sort_values(["MAE","RMSE","key"]).drop_duplicates("feature_set"))
assert set(rq4_by_feature.feature_set) == set(feature_sets)
# All saved candidate pipelines were trained on TRAIN only; notebook 03 refits on train+validation.
selected_keys = sorted(set([str(winner.key)] + best_by_algorithm.key.tolist() + rq4_by_feature.key.tolist()))
for i,key in enumerate(selected_keys):
    joblib.dump(fitted[key],OUT/f"train_pipeline_{i:02d}.joblib")
lookup = {key:f"train_pipeline_{i:02d}.joblib" for i,key in enumerate(selected_keys)}
selection = {
 "seed":SEED,"criterion":"Validation MAE, then RMSE; no test selection",
 "source_sha256":actual_hash,"feature_sets":feature_sets,
 "primary":{"key":str(winner.key),"algorithm":selected_algorithm,"feature_set":str(winner.feature_set),
            "params":json.loads(winner.params_json)},
 "rq3_keys":best_by_algorithm.set_index("algorithm").key.to_dict(),
 "rq4_keys":rq4_by_feature.set_index("feature_set").key.to_dict(),
 "candidate_artifacts":lookup,
 "train_target_quarters":sorted(qsets["train"]),
 "validation_target_quarters":sorted(qsets["validation"]),
 "test_target_quarters":sorted(qsets["test"]),
 "python_version":platform.python_version(),
 "note":"All choices frozen using development data; test outcomes not used."
}
(OUT/"selection_manifest.json").write_text(json.dumps(selection,indent=2))
display(pd.DataFrame([selection["primary"]]))
display(best_by_algorithm[["algorithm","MAE","RMSE","R2"]])
display(rq4_by_feature[["feature_set","MAE","RMSE","R2"]])
print("Frozen selection:",OUT/"selection_manifest.json")

## Limitations and next steps

- Selection uses limited validation quarters; a broader rolling-origin sensitivity is desirable where sufficient historical periods exist.
- Extremely large PAT growth values may distort MAE/RMSE; define sensitivity analysis *before* inspecting test outcomes.
- The current file's financial values may include filing-availability leakage unless filing/release dates are independently verified.
- Notebook 03 must report all held-out outcomes, including unfavorable ones, and cannot change the selection retrospectively.
- Research Question 3 is answered using **all frozen algorithm comparators**, not only the primary winner; Question 4 uses matched feature-set comparisons.